Modelado de Línea Base (Baselines) y Vectorización Clásica

El objetivo de este cuaderno es establecer el rendimiento mínimo exigible (baseline) para el problema de clasificación de 55 tripletas. Si un modelo complejo (RoBERTa) no es capaz de batir con margen a estos modelos clásicos, la arquitectura profunda no estará justificada a nivel de negocio.

Para los algoritmos estadísticos (Regresión Logística y Random Forest), sí es estrictamente necesario aplicar limpieza de ruido gramatical y vectorización discreta. Se utilizará TF-IDF extrayendo unigramas y bigramas, descartando palabras vacías (stopwords). 

El vector objetivo será la tripleta y la estrategia de partición respetará los clústeres calculados en la Fase 1 mediante GroupKFold para impedir la fuga de datos.

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Ingesta estricta de la Capa Gold
RUTA_GOLD = Path("../data/gold/corpus_sitor_limpio.parquet")

if not RUTA_GOLD.exists():
    raise FileNotFoundError("El corpus gold no existe. Debes ejecutar el notebook 01 previamente.")

df_gold = pd.read_parquet(RUTA_GOLD)

print("--- Auditoría de Ingesta Gold ---")
print(f"Total de registros: {len(df_gold)}")
print(f"Clases a predecir: {df_gold['target_tripleta'].nunique()}")
print(f"Clústeres (Grupos independientes): {df_gold['cluster_id'].nunique()}")

# Codificación del Target (Etiquetas de texto a enteros)
le = LabelEncoder()
y_encoded = le.fit_transform(df_gold['target_tripleta'])
grupos = df_gold['cluster_id']

# Vectorización Matemática para ML Clásico
print("\nTransformando texto a espacio vectorial TF-IDF...")
vectorizador = TfidfVectorizer(
    max_features=5000, 
    stop_words='english', 
    ngram_range=(1, 2)
)

X_tfidf = vectorizador.fit_transform(df_gold['full_text'])

print(f"Dimensiones de Matriz X (Features): {X_tfidf.shape}")
print(f"Dimensiones de Vector y (Target): {y_encoded.shape}")

--- Auditoría de Ingesta Gold ---
Total de registros: 15403
Clases a predecir: 56
Clústeres (Grupos independientes): 15241

Transformando texto a espacio vectorial TF-IDF...
Dimensiones de Matriz X (Features): (15403, 5000)
Dimensiones de Vector y (Target): (15403,)


Evaluación Robusta de Línea Base (Baselines)

Para obtener una estimación insesgada del rendimiento de los algoritmos clásicos, se implementa una validación cruzada anidada lógica mediante `StratifiedGroupKFold` (5 particiones). 

Este particionador resuelve los dos problemas estructurales del dataset de forma simultánea:
* `Stratified`: Mantiene la proporción de las 56 clases en cada fold, vital dado el extremo desbalanceo.
* `Group`: Obliga a que todos los tickets de un mismo `cluster_id` (plantilla) caigan en la misma partición (entrenamiento o validación), cerrando definitivamente cualquier fuga de datos por similitud léxica.

Se evalúa Regresión Logística y Random Forest penalizando a las clases mayoritarias (`class_weight='balanced'`) y monitorizando la métrica F1-Macro, que es el indicador real de rendimiento en matrices desbalanceadas.

In [2]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import warnings

# Suprimimos warnings de clases raras cayendo en un solo fold temporalmente
warnings.filterwarnings('ignore', category=UserWarning)

print("Inicializando StratifiedGroupKFold (K=5)...")
cv_strategy = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    "Regresion_Logistica": LogisticRegression(
        class_weight='balanced', 
        max_iter=1000, 
        random_state=42,
        n_jobs=-1
    ),
    "Random_Forest": RandomForestClassifier(
        n_estimators=100, 
        class_weight='balanced', 
        max_depth=30, # Acotamos profundidad para evitar overfitting masivo en TF-IDF
        random_state=42,
        n_jobs=-1
    )
}

resultados_cv = []

for nombre, modelo in modelos.items():
    print(f"\nEjecutando Cross-Validation para: {nombre}...")
    
    scores = cross_validate(
        estimator=modelo, 
        X=X_tfidf, 
        y=y_encoded, 
        groups=grupos, 
        cv=cv_strategy,
        scoring=['accuracy', 'f1_macro'],
        return_train_score=False,
        n_jobs=1 # Ejecución secuencial de particiones para proteger la RAM local
    )
    
    acc_mean = np.mean(scores['test_accuracy'])
    f1_mean = np.mean(scores['test_f1_macro'])
    
    print(f"[{nombre}] Accuracy: {acc_mean:.4f} | F1-Macro: {f1_mean:.4f}")
    
    resultados_cv.append({
        'Modelo': nombre,
        'Accuracy': acc_mean,
        'F1_Macro': f1_mean
    })

df_resumen = pd.DataFrame(resultados_cv).sort_values(by='F1_Macro', ascending=False)
print("\n--- Rendimiento Definitivo de Línea Base ---")
display(df_resumen)

Inicializando StratifiedGroupKFold (K=5)...

Ejecutando Cross-Validation para: Regresion_Logistica...


d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.

[Regresion_Logistica] Accuracy: 0.3543 | F1-Macro: 0.4071

Ejecutando Cross-Validation para: Random_Forest...
[Random_Forest] Accuracy: 0.3923 | F1-Macro: 0.4395

--- Rendimiento Definitivo de Línea Base ---


,Modelo,Accuracy,F1_Macro
1,Random_Forest,0.392325,0.439457
0,Regresion_Logistica,0.354281,0.407052


Auditoría de Degradación por Clases

Para documentar empíricamente las limitaciones de los enfoques basados en bolsa de palabras (Bag of Words / TF-IDF) frente a problemas de alta cardinalidad, se aísla una partición estática de validación (Hold-out 80/20) respetando la agrupación por plantillas. 

Se evalúa el modelo ganador (Random Forest) generando un reporte de clasificación detallado. El objetivo es evidenciar el colapso de la métrica F1-Score en las tripletas minoritarias y justificar la transición a arquitecturas de Deep Learning con comprensión semántica.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

# Extraemos una partición estática 80/20 protegiendo los clústeres
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X_tfidf, y_encoded, groups=grupos))

X_train, X_test = X_tfidf[train_idx], X_tfidf[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

print(f"Entrenando Random Forest de auditoría (Train: {len(y_train)} | Test: {len(y_test)})...")

rf_audit = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', 
    max_depth=30,
    random_state=42,
    n_jobs=-1
)

rf_audit.fit(X_train, y_train)
y_pred = rf_audit.predict(X_test)

# Mapeamos las clases que realmente existen en el set de test para evitar desajustes
clases_en_test = np.unique(y_test)
nombres_clases = le.inverse_transform(clases_en_test)

reporte = classification_report(
    y_test, 
    y_pred, 
    target_names=nombres_clases,
    zero_division=0
)

print("\n--- Reporte de Clasificación Detallado (Random Forest) ---")
print(reporte)

Entrenando Random Forest de auditoría (Train: 12323 | Test: 3080)...

--- Reporte de Clasificación Detallado (Random Forest) ---
                                                 precision    recall  f1-score   support

               Billing and Payments_Change_high       0.93      0.88      0.90        16
                Billing and Payments_Change_low       1.00      1.00      1.00         7
             Billing and Payments_Change_medium       0.57      0.67      0.62         6
             Billing and Payments_Incident_high       0.67      0.56      0.61        39
              Billing and Payments_Incident_low       0.75      0.52      0.62        23
           Billing and Payments_Incident_medium       0.49      0.58      0.53        50
              Billing and Payments_Problem_high       0.69      0.75      0.72        32
               Billing and Payments_Problem_low       0.72      0.57      0.64        37
            Billing and Payments_Problem_medium       0.58      0.51 